# EVA — Обучение в Google Colab
233M параметров, 1 слой, Cosine annealing + gradient accumulation


In [ ]:
# 1. Клонируем репозиторий
!git clone https://github.com/BlackCatSpb/FCF.git /content/EVA
%cd /content/EVA
!pip install -r requirements.txt -q
!pip install faiss-cpu -q

In [ ]:
# 2. Монтируем Google Drive для сохранения снапшотов
from google.colab import drive
drive.mount('/content/drive')

# Создаём папки для снапшотов и весов
!mkdir -p /content/drive/MyDrive/EVA/snapshots
!mkdir -p /content/drive/MyDrive/EVA/checkpoints

In [ ]:
# 3. Проверяем GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# 4. Запускаем обучение
import sys
sys.path.insert(0, '/content/EVA')

from eva.config import FCFConfig
from eva.primordial_layer import PrimordialLayer
from eva.tokenizer_utils import load_or_create_tokenizer
from eva.language_trainer import LanguageTrainer
from eva.unified_grammar import UnifiedStateGrammar
from eva.utils import save_primordial_layer
import os
import torch
import json
import time

# Конфиг
config = FCFConfig()
config.training.learning_rate = 1e-4
config.training.max_steps = 50000  # Больше шагов для лучшего качества

# Создаём слой
layer = PrimordialLayer(config)
layer = layer.cuda()
print(f"Создан: {layer.summary()}")

# Токенизатор
tokenizer = load_or_create_tokenizer()

# Grammar
grammar = UnifiedStateGrammar(config.d_model)

# Трейнер
trainer = LanguageTrainer(
    layer=layer,
    tokenizer=tokenizer,
    config=config,
    checkpoint_dir='/content/drive/MyDrive/EVA/checkpoints',
    state_grammar=grammar,
    benchmark_interval=1000,
)

# Увеличиваем интервал чекпоинтов (сохраняем только снапшоты, не веса)
trainer.checkpoint_interval = 5000  # Веса сохраняем редко
trainer.gen_test_interval = 1000    # Тест генерации чаще

# Датасет
train_file = '/content/EVA/real_data/combined_ru.txt'

# Кастомный цикл обучения с сохранением только снапшотов
def train_with_snapshots(trainer, max_steps, device, text_file):
    """Обучение с сохранением снапшотов каждые 500 шагов, весов — каждые 5000."""
    import pickle
    import faiss
    
    snapshot_dir = '/content/drive/MyDrive/EVA/snapshots'
    os.makedirs(snapshot_dir, exist_ok=True)
    
    # Запускаем стандартное обучение, но переопределяем _save_checkpoint
    original_save = trainer._save_checkpoint
    
    def lightweight_save(final=False):
        """Сохраняем только снапшоты и метаданные, не веса."""
        path = os.path.join(snapshot_dir, f"step_{trainer.step:06d}" if not final else "final")
        os.makedirs(path, exist_ok=True)
        
        # Сохраняем снапшоты (FAISS + meta)
        with open(os.path.join(path, 'snapshots.pkl'), 'wb') as f:
            pickle.dump(trainer.layer.state_storage.snapshots_meta, f)
        
        if trainer.layer.state_storage.index is not None:
            faiss.write_index(trainer.layer.state_storage.index, os.path.join(path, 'index.faiss'))
        
        with open(os.path.join(path, 'meta.pkl'), 'wb') as f:
            pickle.dump({
                'usage_count': trainer.layer.meta.usage_count,
                'confidence_history': trainer.layer.meta.confidence_history,
                'created_at': trainer.layer.meta.created_at,
            }, f)
        
        # Сохраняем конфиг
        trainer.layer.config.to_json(os.path.join(path, 'config.json'))
        
        # Веса сохраняем ТОЛЬКО если final или каждые 5000 шагов
        if final or trainer.step % 5000 == 0:
            torch.save(trainer.layer.state_dict(), os.path.join(path, 'weights.pt'))
            print(f"[Save] Веса сохранены: {path}/weights.pt")
        else:
            # Создаём symlink на последние веса вместо копирования
            latest = os.path.join(snapshot_dir, 'latest_weights.pt')
            if os.path.exists(latest):
                os.remove(latest)
            torch.save(trainer.layer.state_dict(), latest)
        
        # Статус
        status = {
            'step': trainer.step,
            'snapshots': len(trainer.layer.state_storage.snapshots_meta),
            'confidence': trainer.layer.meta.average_confidence(),
            'timestamp': time.time(),
        }
        with open(os.path.join(path, 'status.json'), 'w') as f:
            json.dump(status, f)
        
        print(f"[Save] step={trainer.step} snapshots={status['snapshots']} conf={status['confidence']:.3f}")
    
    trainer._save_checkpoint = lightweight_save
    
    # Запускаем обучение
    stats = trainer.train(
        max_steps=max_steps,
        device=device,
        text_file=text_file,
        block_size=512,
        auto_stop=True,
    )
    
    return stats

print("\n" + "="*60)
print("  EVA — Обучение с сохранением снапшотов")
print("="*60)
print(f"  Макс. шагов: 50000")
print(f"  Снапшоты: каждые 500 шагов")
print(f"  Веса: каждые 5000 шагов + финал")
print("="*60 + "\n")

stats = train_with_snapshots(trainer, 50000, 'cuda', train_file)
print(f"\nОбучение завершено: {stats}")

In [ ]:
# 5. Тест генерации после обучения
import torch

prompts = [
    "История это наука которая изучает",
    "Математика помогает человечеству",
    "Природа Земли удивительна потому что",
    "Компьютеры обрабатывают данные с помощью",
    "Человек отличается от животных тем что",
]

layer.eval()
for prompt in prompts:
    encoding = tokenizer.encode(prompt)
    ids = encoding.ids if hasattr(encoding, 'ids') else encoding
    input_ids = torch.tensor([ids], dtype=torch.long).cuda()
    output = layer.generate(input_ids, max_new_tokens=80, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(output[0].tolist())
    print(f"\nQ: {prompt}")
    print(f"A: {response}")

# Сохраняем тестовые результаты
import json
results = []
layer.eval()
for prompt in prompts:
    encoding = tokenizer.encode(prompt)
    ids = encoding.ids if hasattr(encoding, 'ids') else encoding
    input_ids = torch.tensor([ids], dtype=torch.long).cuda()
    output = layer.generate(input_ids, max_new_tokens=80, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(output[0].tolist())
    results.append({"Q": prompt, "A": response})

with open('/content/drive/MyDrive/EVA/generation_test.json', 'w', encoding='utf-8') as f:
    json.dump({"step": trainer.step, "results": results}, f, ensure_ascii=False, indent=2)
print("\nРезультаты сохранены в Google Drive")

In [ ]:
# 6. Упаковка для скачивания
import os

# Проверяем что есть
print("=== Файлы в Google Drive ===")
for root, dirs, files in os.walk('/content/drive/MyDrive/EVA'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {f}: {size:.1f} MB")

# Упаковываем только снапшоты и конфиг (без весов)
!cd /content/drive/MyDrive/EVA && tar czf /content/eva_snapshots.tar.gz snapshots/

# Скачиваем снапшоты
from google.colab import files
print("\nСкачиваем снапшоты...")
files.download('/content/eva_snapshots.tar.gz')

# Веса скачиваем отдельно (большой файл)
print("\nСкачиваем веса (~932 MB)...")
files.download('/content/drive/MyDrive/EVA/snapshots/latest_weights.pt')

In [ ]:
# 8. Продолжение обучения (если Colab отключился)
import sys
sys.path.insert(0, '/content/EVA')

from eva.config import FCFConfig
from eva.primordial_layer import PrimordialLayer
from eva.tokenizer_utils import load_or_create_tokenizer
from eva.language_trainer import LanguageTrainer
from eva.unified_grammar import UnifiedStateGrammar
from eva.utils import load_primordial_layer

# Загружаем последние веса
weights_path = '/content/drive/MyDrive/EVA/snapshots/latest_weights.pt'
config = FCFConfig()
layer = PrimordialLayer(config)

import torch
layer.load_state_dict(torch.load(weights_path, map_location='cpu'))
layer = layer.cuda()
print(f"Загружены веса из: {weights_path}")

# Токенизатор
tokenizer = load_or_create_tokenizer()

# Grammar
grammar = UnifiedStateGrammar(config.d_model)

# Трейнер
trainer = LanguageTrainer(
    layer=layer,
    tokenizer=tokenizer,
    config=config,
    checkpoint_dir='/content/drive/MyDrive/EVA/checkpoints',
    state_grammar=grammar,
    benchmark_interval=1000,
)

# Загружаем снапшоты
import pickle
import faiss
import os

snapshot_dir = '/content/drive/MyDrive/EVA/snapshots/final'
if os.path.exists(snapshot_dir):
    with open(os.path.join(snapshot_dir, 'snapshots.pkl'), 'rb') as f:
        layer.state_storage.snapshots_meta = pickle.load(f)
    layer.state_storage.rebuild_from_meta()
    
    index_path = os.path.join(snapshot_dir, 'index.faiss')
    if os.path.exists(index_path):
        layer.state_storage.index = faiss.read_index(index_path)
    
    with open(os.path.join(snapshot_dir, 'meta.pkl'), 'rb') as f:
        meta = pickle.load(f)
        layer.meta.usage_count = meta.get('usage_count', 0)
        layer.meta.confidence_history = meta.get('confidence_history', [])
    
    print(f"Загружены снапшоты: {len(layer.state_storage.snapshots_meta)} шт.")

# Продолжаем обучение
train_file = '/content/EVA/real_data/combined_ru.txt'
stats = trainer.train(
    max_steps=50000,  # Общее количество шагов
    device='cuda',
    text_file=train_file,
    block_size=512,
    auto_stop=True,
)
print(f"Обучение завершено: {stats}")

In [ ]:
# 9. Инструкция: как загрузить обученную модель локально
print("""
=== ИНСТРУКЦИЯ ПО ЗАГРУЗКЕ МОДЕЛИ ЛОКАЛЬНО ===

1. Скачайте:
   - eva_snapshots.tar.gz (снапшоты, ~100 MB)
   - latest_weights.pt (веса, ~932 MB)

2. Распакуйте снапшоты:
   tar xzf eva_snapshots.tar.gz -C /path/to/EVA/

3. Положите веса:
   mkdir -p /path/to/EVA/checkpoints/lazy/final
   cp latest_weights.pt /path/to/EVA/checkpoints/lazy/final/weights.pt

4. Скопируйте снапшоты:
   cp -r snapshots/final/* /path/to/EVA/checkpoints/lazy/final/

5. Запустите локально:
   cd /path/to/EVA
   python run.py --lazy-learn --checkpoint checkpoints/lazy/final

Модель продолжит обучение с того же места,
используя снапшоты для самоорганизации.
""")